# AICE Associate 변주 문제 - Predicted_CLV,Customer_Churn_Risk 예측 (회귀)

### 이 노트북의 구성요소

아래 다이어그램은 이 노트북 해설에서 실제로 사용된 함수·클래스·모듈을 구간별로 모은 것입니다 (굵게 표시된 이름은 이 노트북에서 특별히 선택된 항목입니다).

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>

<div style="display:flex; flex-direction:column; gap:6px; width:fit-content;">
<pre class="mermaid">
flowchart LR
    subgraph LIB["라이브러리"]
        LIB1["<b>라이브러리</b><br/>sklearn"]
    end
    subgraph EXP["데이터 탐색"]
        EXP1["<b>데이터 탐색</b><br/>head<br/>read_csv"]
    end
    subgraph PREP["전처리"]
        PREP1["<b>전처리</b><br/><b>MinMaxScaler</b><br/>LabelEncoder<br/>drop<br/>dropna<br/>train_test_split"]
    end
    LIB --> EXP --> PREP
    classDef libCls fill:#F0EEE8,stroke:#8A8478,color:#3A362E
    class LIB libCls
    classDef expCls fill:#E9F1F6,stroke:#5B7C99,color:#1E3548
    class EXP expCls
    classDef prepCls fill:#FAF3E7,stroke:#B08D57,color:#5C4720
    class PREP prepCls
</pre>
<pre class="mermaid">
flowchart LR
    subgraph ML["머신러닝"]
        ML1["<b>머신러닝</b><br/><b>GradientBoostingRegressor</b><br/><b>mean_squared_error</b><br/>DataFrame<br/>DecisionTreeRegressor<br/>GridSearchCV<br/>sort_values"]
    end
    subgraph UNSUP["비지도학습"]
        UNSUP1["<b>비지도학습</b><br/>(기본 구성)"]
    end
    subgraph DL["딥러닝"]
        DL1["<b>딥러닝</b><br/>(기본 구성)"]
    end
    ML --> UNSUP --> DL
    classDef mlCls fill:#FBEEEA,stroke:#B5654A,color:#5C2A1B
    class ML mlCls
    classDef unsupCls fill:#F3EEF4,stroke:#8B6F8E,color:#402F42
    class UNSUP unsupCls
    classDef dlCls fill:#EBEEF6,stroke:#445577,color:#232C3D
    class DL dlCls
</pre>
</div>

<script>
mermaid.initialize({ startOnLoad: false, securityLevel: "loose" });
mermaid.run();
</script>
"""))

### 시나리오

스타트업 '패션링크(FashionLink)'는 최근 신규 고객 유입에 따른 마케팅 비용 증가를 관리하고, 장기적인 수익성을 극대화하기 위해 이탈 가능성이 높은 고객을 선제적으로 예측하고자 합니다. 이 모델을 통해 이탈 위험이 높은 고객 그룹을 식별하고, 맞춤형 유지(Retention) 캠페인을 실행하여 고객 생애 가치(LTV)를 최대화하는 것을 목표로 합니다.

---

**[유의사항]**
- 답안은 각 문항 아래 표시된 `# (N) 여기에 ...` 칸에 작성하세요.
- **정답/해설은 이 노트북 가장 아래 `## 해설` 섹션에 모아뒀습니다.** 먼저 스스로 풀어본 뒤에 확인하세요.
- 이 노트북은 오리지널 창작 문제이며, 실제 AICE 샘플문항 원문을 복제하지 않습니다.

**[데이터 컬럼 설명]**

- Predicted_CLV : Predicted_CLV,Customer_Churn_Risk
- Total_Purchase_Amount : 고객이 특정 기간 동안 지출한 총 구매 금액 (단위: KRW)
- Customer_Segment : 고객의 세그먼트 분류 (예: 신규 고객, 충성 고객, 이탈 위험군, VIP)
- Days_Since_Last_Purchase : 피처 컬럼
- Average_Order_Value : 피처 컬럼
- Product_Category_Preference : 피처 컬럼
- Membership_Tier : 피처 컬럼
- Customer_ID : 식별자(모델링에 불필요)

## 0. 데이터 준비

다음 문항을 풀기 전에 아래 코드를 실행하세요(문제 데이터를 생성합니다).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def synthesize_dataframe(seed=0, n_rows=600):
    rng = np.random.default_rng(seed)
    n = n_rows

    base = rng.normal(loc=50, scale=15, size=n)
    outlier_idx = rng.choice(n, size=max(3, n // 50), replace=False)
    base[outlier_idx] += rng.choice([1, -1], size=len(outlier_idx)) * rng.uniform(80, 150, size=len(outlier_idx))

    df = pd.DataFrame({'Total_Purchase_Amount': base.round(2)})
    n_cats = rng.integers(3, 5)
    cats = [f"Type{i+1}" for i in range(n_cats)]
    df['Customer_Segment'] = rng.choice(cats, size=n)

    for col in ['Days_Since_Last_Purchase', 'Average_Order_Value', 'Product_Category_Preference', 'Membership_Tier']:
        df[col] = rng.normal(0, 1, size=n).round(2)

    df['Customer_ID'] = [f"ID{i:05d}" for i in range(n)]

    z = (base - base.mean()) / (base.std() + 1e-9)
    if '회귀' == "분류":
        prob = 1 / (1 + np.exp(-z))
        labels = (rng.random(n) < prob).astype(int)
        classes = [c.strip() for c in 'Predicted_CLV,Customer_Churn_Risk'.split(",") if c.strip()][:2]
        if len(classes) < 2:
            classes = ["Positive", "Negative"]
        df['Predicted_CLV'] = np.where(labels == 1, classes[0], classes[-1])
    else:
        noise = rng.normal(0, 5, size=n)
        df['Predicted_CLV'] = (base * 1.5 + noise).round(2)

    for col in ['Total_Purchase_Amount'] + ['Days_Since_Last_Purchase', 'Average_Order_Value', 'Product_Category_Preference', 'Membership_Tier'][:2]:
        na_idx = rng.choice(n, size=int(n * 0.03), replace=False)
        df.loc[na_idx, col] = np.nan

    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


data = synthesize_dataframe(seed=1003)
data.to_csv("data.csv", index=False)
print("data.csv 저장 완료 -", data.shape)
data.head(4)

## <데이터 분석>

### 1. 라이브러리 임포트

scikit-learn 을 alias **sk** 로 임포트하세요.

In [ ]:
# (1) 여기에 답안코드를 작성하고 실행하세요



### 2. 데이터 로드

pandas 로 'data.csv' 를 읽어와 데이터프레임 변수명 **my_data** 에 할당하고, 첫 4개 행을 출력하세요.

In [ ]:
# (2) 여기에 답안코드를 작성하고 실행하세요



### 3. 시각화 (subplots)

고객의 세그먼트 분류 (예: 신규 고객, 충성 고객, 이탈 위험군, VIP)(Customer_Segment) 분포의 countplot 과, Predicted_CLV,Customer_Churn_Risk 별 고객이 특정 기간 동안 지출한 총 구매 금액 (단위: KRW)(Total_Purchase_Amount) histplot 을 나란히 그리는 코드입니다.
빈칸 **(A)** 에 들어갈, 여러 그래프를 한 번에 그릴 때 쓰는 matplotlib 함수 이름은?

```python
fig, axes = plt.(A)(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=my_data, x='Customer_Segment', ax=axes[0])
sns.histplot(data=my_data, x='Total_Purchase_Amount', hue='Predicted_CLV', ax=axes[1])
plt.show()
```

In [ ]:
# (3) 여기에 답을 입력하세요 (실행 불필요)



### 4. 시각화 (boxplot)

Predicted_CLV,Customer_Churn_Risk 별 고객이 특정 기간 동안 지출한 총 구매 금액 (단위: KRW)(Total_Purchase_Amount) 의 분포를 seaborn boxplot 으로 시각화하세요 (X축: 타깃, Y축: 수치형 컬럼).

In [ ]:
# (4) 여기에 답안코드를 작성하고 실행하세요



## <데이터 전처리>

### 5. 이상치 처리

고객이 특정 기간 동안 지출한 총 구매 금액 (단위: KRW)(Total_Purchase_Amount) 컬럼의 IQR 기준(K=1.5)을 벗어나는 이상치 행을 제거하고, 불필요한 컬럼(Customer_ID)도 삭제해서 **data_temp** 에 저장하는 코드입니다. 빈칸 **(A)** 에 들어갈, 행을 삭제할 때 쓰는 DataFrame 메서드 이름은?

```python
q1 = my_data['Total_Purchase_Amount'].quantile(0.25)
q3 = my_data['Total_Purchase_Amount'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
data_temp = my_data.(A)(my_data[(my_data['Total_Purchase_Amount'] > upper_fence) | (my_data['Total_Purchase_Amount'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['Customer_ID'])
data_temp = data_temp.reset_index(drop=True)
```

In [ ]:
# (5) 여기에 답을 입력하세요 (실행 불필요)



### 6. 결측치 처리

다음은 data_temp 의 결측치를 처리하는 코드인데, 실행하면 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
print('결측치 처리 전')
print(data_temp.isnull().sum())
data_na = data_temp.dropna()
print('결측치 처리 후')
print(data_na.isnull().sum())
# (이 버전에는 의도한 것과 다른 결과를 내는 부분이 있습니다)
```

In [ ]:
# (6) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 7. 인코딩

고객의 세그먼트 분류 (예: 신규 고객, 충성 고객, 이탈 위험군, VIP)(Customer_Segment) 컬럼을 sklearn 의 **LabelEncoder** 로 인코딩해서 data_preset 에 저장하세요.

In [ ]:
# (7) 여기에 답안코드를 작성하고 실행하세요



### 8. 데이터 분리

Predicted_CLV 을 y, 나머지를 X 로 삼아 train_test_split 으로 분리하세요.
- test_size=0.3, random_state=7
- 변수명: X_train, X_valid, y_train, y_valid

In [ ]:
# (8) 여기에 답안코드를 작성하고 실행하세요



### 9. 스케일링

MinMaxScaler 로 훈련/검증 데이터를 스케일링하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
if len(X_valid) > 0:
    X_valid_scaled = scaler.transform(X_valid)
else:
    X_valid_scaled = None
```

In [ ]:
# (9) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



## <AI 모델링>

### 10. GridSearch 모델링

DecisionTreeRegressor 와 GradientBoostingRegressor 를 GridSearchCV(cv=3)로 탐색하고 학습하세요.
- max_depth 후보: [3, 5, 7]
- GradientBoostingRegressor 의 n_estimators 후보: [50, 100]
- 변수명: gs_a (트리 모델), gs_b (앙상블 모델)

In [ ]:
# (10) 여기에 답안코드를 작성하고 실행하세요



### 11. GridSearch 결과 확인

위 GridSearch에서 GradientBoostingRegressor 의 n_estimators 후보는 [50, 100] 였습니다. GridSearchCV가 고를 수 있는 값의 후보 중 '가장 큰 값'은 얼마인가요?

In [ ]:
# (11) 여기에 답을 입력하세요 (실행 불필요)



### 12. 변수중요도

GradientBoostingRegressor 의 변수중요도 Top 10개를 뽑아 시각화하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
fi = pd.DataFrame({'feature': gs_b.best_estimator_.feature_importances_, 'importance': X_train.columns})
fi = fi.sort_values('importance', ascending=False)[:10]
sns.barplot(x='importance', y='feature', data=fi)
plt.show()
```

In [ ]:
# (12) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 13. 성능평가

검증데이터로 gs_a, gs_b 두 모델의 **mean_squared_error** 를 각각 계산해서 a_score, b_score 에 저장하세요.

In [ ]:
# (13) 여기에 답안코드를 작성하고 실행하세요



---
## 해설

스스로 풀어본 뒤 아래에서 확인하세요. 문항 번호가 위 문제 번호와 일치합니다.

### 1번 해설 - 라이브러리 임포트 [코드작성]

In [ ]:
import sklearn as sk

> import ... as ... 문법으로 별칭을 지정합니다.

### 2번 해설 - 데이터 로드 [코드작성]

In [ ]:
my_data = pd.read_csv('data.csv')
my_data.head(4)

> pd.read_csv() 로 파일을 읽고 head() 로 상위 행을 확인합니다.

### 3번 해설 - 시각화 (subplots) [빈칸채우기]

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=my_data, x='Customer_Segment', ax=axes[0])
sns.histplot(data=my_data, x='Total_Purchase_Amount', hue='Predicted_CLV', ax=axes[1])
plt.show()

> plt.subplots() 는 nrows/ncols 로 여러 축(Axes)을 한 번에 만듭니다. 정답: subplots

### 4번 해설 - 시각화 (boxplot) [코드작성]

In [ ]:
sns.boxplot(data=my_data, x='Predicted_CLV', y='Total_Purchase_Amount')
plt.show()

> sns.boxplot(x=타깃, y=수치형컬럼) 형태로 그립니다.

### 5번 해설 - 이상치 처리 [빈칸채우기]

In [ ]:
q1 = my_data['Total_Purchase_Amount'].quantile(0.25)
q3 = my_data['Total_Purchase_Amount'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
data_temp = my_data.drop(my_data[(my_data['Total_Purchase_Amount'] > upper_fence) | (my_data['Total_Purchase_Amount'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['Customer_ID'])
data_temp = data_temp.reset_index(drop=True)

> DataFrame.drop() 은 행(기본 axis=0) 또는 열(axis=1)을 삭제합니다. 정답: drop

### 6번 해설 - 결측치 처리 [오류정정]

In [ ]:
print('결측치 처리 전')
print(data_temp.isnull().sum())
data_na = data_temp.dropna()
print('결측치 처리 후')
print(data_na.isnull().sum())

> [off_by_one] 오프바이원(Off-by-one) 오류: 인덱스 범위 초과, 반복/개수가 1 부족·초과되는 값 사용

### 7번 해설 - 인코딩 [코드작성]

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data_preset = data_na.copy()
data_preset['Customer_Segment'] = le.fit_transform(data_preset['Customer_Segment'])

> 범주형 데이터를 숫자로 바꾸는 두 가지 방법 중 하나를 사용합니다.

### 8번 해설 - 데이터 분리 [코드작성]

In [ ]:
from sklearn.model_selection import train_test_split

X = data_preset.drop('Predicted_CLV', axis=1)
y = data_preset['Predicted_CLV']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=7)

> train_test_split(X, y, ...) 은 X_train, X_valid, y_train, y_valid 순서로 반환합니다.

### 9번 해설 - 스케일링 [오류정정]

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

> [loop_control] 원래 코드는 항상 X_valid에 대해 transform을 수행했지만, 버그 코드에서는 X_valid의 길이가 0일 경우 X_valid_scaled를 None으로 설정하여 로직 흐름을 변경했습니다. 이는 scaler 객체를 사용하여 모든 데이터셋에 일관되게 적용되어야 하는 일반적인 스케일링 절차에서 조건문의 부적절한 사용으로 인해 발생할 수 있는 제어 흐름 오류입니다.

### 10번 해설 - GridSearch 모델링 [코드작성]

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

gs_a = GridSearchCV(DecisionTreeRegressor(random_state=7), {'max_depth':[3,5,7]}, cv=3)
gs_a.fit(X_train_scaled, y_train)

gs_b = GridSearchCV(GradientBoostingRegressor(random_state=7), {'n_estimators':[50, 100], 'max_depth':[3,5,7]}, cv=3)
gs_b.fit(X_train_scaled, y_train)

> GridSearchCV(estimator, param_grid, cv=...).fit(X_train, y_train) 형태로 탐색합니다.

### 11번 해설 - GridSearch 결과 확인 [결과값예측]

In [ ]:
100

> 제시된 후보 중 GridSearch가 고를 수 있는 최댓값을 묻는 문항입니다.

### 12번 해설 - 변수중요도 [오류정정]

In [ ]:
fi = pd.DataFrame({'feature': X_train.columns, 'importance': gs_b.best_estimator_.feature_importances_})
fi = fi.sort_values('importance', ascending=False)[:10]
sns.barplot(x='importance', y='feature', data=fi)
plt.show()

> [scope_reference] DataFrame을 생성할 때 feature와 importance의 데이터 소스를 서로 잘못 교차하여 할당했습니다. 결과적으로 특성 이름과 중요도 값이 뒤바뀌어 시각화가 잘못된 결과를 초래합니다.

### 13번 해설 - 성능평가 [코드작성]

In [ ]:
from sklearn.metrics import mean_squared_error

y_pred_a = gs_a.best_estimator_.predict(X_valid_scaled)
y_pred_b = gs_b.best_estimator_.predict(X_valid_scaled)

a_score = mean_squared_error(y_valid, y_pred_a)
b_score = mean_squared_error(y_valid, y_pred_b)
print(a_score, b_score)

> sklearn.metrics.mean_squared_error 에 (실제값, 예측값) 순서로 인자를 넣습니다.